# Libraries

In [ ]:
using CategoricalArrays
using CSV
using DataFrames
using DataFramesMeta
using LinearAlgebra
using Statistics
using StatsPlots
using Plots

In [ ]:
cols = distinguishable_colors(15, [RGB(1, 1, 1), RGB(0, 0, 0)], dropseed=true)

## Function for plotting time series

In [ ]:
# This funciton takes a dataframe and smooths a feature over a moving window of years.

function smooth_ts_feat(df_1::DataFrame, feat::String; st_year=1600, end_year=1960, y_ws=20)
    feature = Symbol(feat)
    year_range = end_year - st_year
    m_ys1 = []
    m_ft1 = []
    q1_ft1 = []
    q2_ft1 = []
    for y in 1:(year_range-y_ws)
        ini_year = 1600 + y - 1
        fin_year = ini_year + y_ws
        s_df = @subset(df_1, :Year .< fin_year, :Year .> ini_year)
        if isempty(s_df)
            continue
        end
        push!(m_ys1, mean(s_df[!, :Year]))
        med_nov = median(s_df[!, feature])
        push!(m_ft1, med_nov)
        push!(q1_ft1, med_nov - quantile(s_df[!, feature], 0.25))
        push!(q2_ft1, quantile(s_df[!, feature], 0.75) - med_nov)

    end
    df1_out = DataFrame(:Year => m_ys1, :Med => m_ft1, :Q1 => q1_ft1, :Q2 => q2_ft1)
    return df1_out
end

In [ ]:
function plot_metric(df1;
    df2=nothing,
    size=(1200, 1000),
    title="",
    ylabel="Metric value",
    xlabel="Year",
    xlims=nothing,
    ylims=(0.3, 1.2),
    legend=:topleft,
    background_color=RGBA(1, 1, 1, 0),
    colors=[:blue, :red],
    labels=["Version 1", "Version 2"],
    vlines=[1700, 1750, 1820, 1890],
    linewidth=6,
    fillalpha=0.3,
    titlefontsize=14,
    xlabelfontsize=12,
    ylabelfontsize=16,
    xtickfontsize=14,
    ytickfontsize=14,
    legendfontsize=10,
    left_margin=20Plots.mm,    # Add left margin
    bottom_margin=25Plots.mm,   # Add bottom margin
    right_margin=25Plots.mm,    # Add right margin
    top_margin=20Plots.mm)      # Add top margin

    # Create the base plot
    plt = plot(
        ylabel=ylabel,
        xlabel=xlabel,
        legend=legend,
        background_color=background_color,
        size=size,
        ylims=ylims,
        title=title,
        titlefontsize=titlefontsize,
        xguidefontsize=xlabelfontsize,
        yguidefontsize=ylabelfontsize,
        xtickfontsize=xtickfontsize,
        ytickfontsize=ytickfontsize,
        legendfontsize=legendfontsize
    )

    # Add xlims if provided
    if xlims !== nothing
        plot!(plt, xlims=xlims)
    end

    # Add vertical lines
    vline!(plt, vlines, linewidth=2, color=:black, label="", linestyle=:dash)

    # Plot first dataframe
    plot!(plt, df1[!, :Year], df1[!, :Med],
        ribbon=[df1[!, :Q1] df1[!, :Q2]],
        linewidth=linewidth,
        linestyle=:solid,
        fillalpha=fillalpha,
        color=colors[1],
        label=labels[1])

    # Plot second dataframe if provided
    if df2 !== nothing
        plot!(plt, df2[!, :Year], df2[!, :Med],
            ribbon=[df2[!, :Q1] df2[!, :Q2]],
            linewidth=linewidth,
            linestyle=:solid,
            fillalpha=fillalpha,
            color=colors[2],
            label=labels[2])
    end

    return plt
end


In [ ]:
pieces_df = CSV.read("Data/Pieces_Metrics.csv", DataFrame) # your Data folder
#removing duplicate
deleteat!(pieces_df, findfirst(pieces_df[!, :Piece_ID] .== "kdf_5095"));

In [ ]:
"""
    get_seq_entropy(sequence; total_elements=24, alpha=1.0)

Calculate the entropy of a sequence with Laplacian smoothing.

# Arguments
- `sequence`: A sequence of elements (vector or array)
- `total_elements`: Total number of possible unique elements (default: 24)
- `alpha`: Laplacian smoothing parameter (default: 1.0)
 
# Returns
- Entropy value (in bits if using log2, nats if using log)

# Details
The function:
1. Counts the frequency of each unique element in the sequence
2. Applies Laplacian smoothing: adds alpha to all counts (observed and unobserved)
3. Calculates probabilities by normalizing the smoothed counts
4. Computes entropy using H = -Σ(p * log2(p))

Laplacian smoothing ensures that elements not present in the sequence still have
a small non-zero probability, preventing log(0) issues and providing a more
robust entropy estimate.
"""
function get_seq_entropy(sequence; total_elements=24, alpha=1.0)
    # Count frequencies of elements in the sequence
    freq_counts = Dict{eltype(sequence), Int}()
    for element in sequence
        freq_counts[element] = get(freq_counts, element, 0) + 1
    end
    
    # Get the number of unique elements observed
    observed_elements = length(freq_counts)
    
    # Apply Laplacian smoothing
    # Add alpha to all observed counts
    smoothed_counts = [count + alpha for count in values(freq_counts)]
    
    # Add alpha for each unobserved element
    unobserved_elements = total_elements - observed_elements
    if unobserved_elements > 0
        append!(smoothed_counts, fill(alpha, unobserved_elements))
    end
    
    # Calculate total count after smoothing
    total_smoothed = sum(smoothed_counts)
    
    # Calculate probabilities
    probabilities = smoothed_counts ./ total_smoothed
    
    # Calculate entropy: H = -Σ(p * log2(p))
    entropy = -sum(p * log2(p) for p in probabilities)
    
    return entropy
end

In [ ]:
function plot_composer_boxplot(df, feature::Symbol, composers::Vector{String}, names::Vector{String};
    size=(1000, 800),
    title="",
    xlabel="",
    ylabel="Composer",
    xlims=nothing,
    legend=false,
    background_color=RGBA(1, 1, 1, 0),
    color=:lightblue,
    linewidth=5,
    #yticks=:auto,
    #xlims=:auto,
    #ylims=:auto,
    titlefontsize=14,
    xlabelfontsize=16,
    ylabelfontsize=16,
    xtickfontsize=10,
    ytickfontsize=16,
    left_margin=20Plots.mm,    # Add left margin
    bottom_margin=40Plots.mm,   # Add bottom margin
    right_margin=25Plots.mm,    # Add right margin
    top_margin=20Plots.mm)      # Add top margin

    # Verify composers and names have same length
    @assert length(composers) == length(names) "composers and names must have same length"

    # Create mapping from composer codes to display names
    composer_to_name = Dict(zip(composers, names))

    # Filter dataframe for specified composers
    df_filtered = @subset(df, :Composer .∈ Ref(composers))

    # Add display name column
    df_filtered = @transform(df_filtered, :ComposerName = [composer_to_name[c] for c in :Composer])

    # Calculate median for each composer and sort ascending
    medians = @combine(groupby(df_filtered, :ComposerName),
        :median_val = median(skipmissing($feature)))
    sort!(medians, :median_val, rev=true)
    name_order = medians.ComposerName

    # Create categorical variable with sorted order
    df_filtered = @transform(df_filtered,
        :Composer_cat = categorical(:ComposerName, levels=name_order))


    # Create VERTICAL boxplot first (which we know works)
    plt = boxplot(df_filtered[!, :Composer_cat], df_filtered[!, feature],
        color=color,
        label="",
        outliers=true,
        xlabel=ylabel,
        ylabel=xlabel,
        legend=legend,
        background_color=background_color,
        size=size,
        title=title,
        titlefontsize=titlefontsize,
        xguidefontsize=ylabelfontsize,
        yguidefontsize=xlabelfontsize,
        xtickfontsize=ytickfontsize,
        ytickfontsize=xtickfontsize,
        ymirror=true,
        yrotation=90,
        xrotation=90,
        left_margin=left_margin,      # Apply margins
        bottom_margin=bottom_margin,
        right_margin=right_margin,
        top_margin=top_margin)

    return plt
end

In [ ]:
"""
    get_composer_feature_stats(df::DataFrame, features::Vector{Symbol}; composers::Union{Vector{String}, Nothing}=nothing)

Calculate mean and median statistics for specified features grouped by composer.

# Arguments
- `df`: DataFrame containing the data
- `features`: Vector of feature column names as Symbols (e.g., [:key_diversity, :mean_unc])
- `composers`: (Optional) Vector of composer names to filter by (matching the :Composer column). 
              If not provided, statistics will be calculated for all unique composers in the dataframe.

# Returns
- DataFrame with composers as rows and mean/median columns for each feature
  Column names follow the pattern: feature_mean, feature_median

# Example
```julia
# For specific composers
stats_df = get_composer_feature_stats(
    pieces_df, 
    [:key_diversity, :mean_unc], 
    composers=["Bach", "Mozart", "Beethoven"]
)

# For all composers
stats_df = get_composer_feature_stats(pieces_df, [:key_diversity, :mean_unc])
```
"""
function get_composer_feature_stats(df::DataFrame, features::Vector{Symbol}; composers::Union{Vector{String}, Nothing}=nothing)
    # If composers not provided, use all unique composers
    if composers === nothing
        composers = unique(df[!, :Composer])
    end
    
    # Filter dataframe for specified composers
    df_filtered = @subset(df, :Composer .∈ Ref(composers))
    
    # Check that all features are numeric
    for feat in features
        if !(string(feat) in names(df_filtered))
            error("Column $feat does not exist in the dataframe")
        end
        col_type = eltype(df_filtered[!, feat])
        if !(col_type <: Union{Number, Missing})
            error("Column $feat has type $col_type, which is not numeric. Only numeric columns can be used for mean/median calculations.")
        end
    end
    
    # Group by composer
    grouped = groupby(df_filtered, :Composer)
    
    # Build aggregation operations manually
    agg_pairs = []
    for feat in features
        mean_name = Symbol(string(feat) * "_mean")
        median_name = Symbol(string(feat) * "_median")
        push!(agg_pairs, feat => (x -> mean(skipmissing(x))) => mean_name)
        push!(agg_pairs, feat => (x -> median(skipmissing(x))) => median_name)
    end
    
    # Add median year as integer
    push!(agg_pairs, :Year => (x -> Int(round(median(skipmissing(x))))) => :Year_median)

    # Calculate statistics using combine
    result_df = combine(grouped, agg_pairs...)
    
    return result_df
end

In [ ]:
function get_quadrant_pieces(df::DataFrame; n_pieces=10)
    # Calculate medians for both features
    median_div = median(df[!, :key_diversity])
    median_unc = median(df[!, :median_unc])

    # Define the four quadrants
    high_div = df[!, :key_diversity] .>= median_div
    low_div = df[!, :key_diversity] .< median_div
    high_unc = df[!, :median_unc] .>= median_unc
    low_unc = df[!, :median_unc] .< median_unc

    # Quadrant 1: High diversity, High uncertainty
    q1_df = df[high_div.&high_unc, :]
    max_div_q1 = maximum(q1_df.key_diversity)
    max_unc_q1 = maximum(q1_df.median_unc)
    distances1 = sqrt.((q1_df.key_diversity .- max_div_q1) .^ 2 .+
                       (q1_df.median_unc .- max_unc_q1) .^ 2)
    q1_sorted = q1_df[sortperm(distances1), :]
    table1 = first(q1_sorted, min(n_pieces, nrow(q1_sorted)))

    # Quadrant 2: High diversity, Low uncertainty
    q2_df = df[high_div.&low_unc, :]
    max_div_q2 = maximum(q2_df.key_diversity)
    min_unc_q2 = minimum(q2_df.median_unc)
    distances2 = sqrt.((q2_df.key_diversity .- max_div_q2) .^ 2 .+
                       (q2_df.median_unc .- min_unc_q2) .^ 2)
    q2_sorted = q2_df[sortperm(distances2), :]
    table2 = first(q2_sorted, min(n_pieces, nrow(q2_sorted)))

    # Quadrant 3: Low diversity, High uncertainty
    q3_df = df[low_div.&high_unc, :]
    min_div_q3 = minimum(q3_df.key_diversity)
    max_unc_q3 = maximum(q3_df.median_unc)
    distances3 = sqrt.((q3_df.key_diversity .- min_div_q3) .^ 2 .+
                       (q3_df.median_unc .- max_unc_q3) .^ 2)
    q3_sorted = q3_df[sortperm(distances3), :]
    table3 = first(q3_sorted, min(n_pieces, nrow(q3_sorted)))

    # Quadrant 4: Low diversity, Low uncertainty
    q4_df = df[low_div.&low_unc, :]
    min_div_q4 = minimum(q4_df.key_diversity)
    min_unc_q4 = minimum(q4_df.median_unc)
    distances4 = sqrt.((q4_df.key_diversity .- min_div_q4) .^ 2 .+
                       (q4_df.median_unc .- min_unc_q4) .^ 2)
    q4_sorted = q4_df[sortperm(distances4), :]
    table4 = first(q4_sorted, min(n_pieces, nrow(q4_sorted)))

    # Return named tuple with all four tables
    return (
        high_div_high_unc=table1,
        high_div_low_unc=table2,
        low_div_high_unc=table3,
        low_div_low_unc=table4
    )
end

In [ ]:
quads = get_quadrant_pieces(pieces_df, n_pieces=15);

In [ ]:
#adding historical periods
m_periods = [minimum(pieces_df[!, :Year]), 1600, 1700, 1750, 1820, 1890, 1960, maximum(pieces_df[!, :Year])+1]
#periods = Vector{String}(undef, size(pieces_df, 1));
#for p in 1:(length(m_periods)-1)
#    ix_p = findall(x -> x >= m_periods[p] && x < m_periods[p+1], pieces_df[!, :Year])
#    periods[ix_p] .= m_per_names[p]
#end

In [ ]:
m_per_names = ["Renaissance", "Early/Mid Baroque", "Late Baroque", "Classical", "Romantic", "Modern", "other"]

# List of composers (labels and names) for the plots in paper

In [ ]:
comp_list = ["bach-js", "frescobaldi", "mozart", "haydn", "beethoven", "handel", "liszt", "dvorak", "schubert", "shostakovitch", "joplin", "couperin", "debussy", "stravinsky"]
comp_names = ["4-Bach", "1-Frescobaldi", "6-Mozart", "5-Haydn", "7-Beethoven", "3-Handel", "9-Liszt", "10-Dvorak", "8-Schubert", "14-Shostakovich", "12-Joplin", "2-Couperin", "13-Debussy", "11-Stravinsky"];

In [ ]:
compo_stats = sort!(get_composer_feature_stats(pieces_df, [:key_diversity, :mean_unc, :Novelty_original, :Novelty_numerals], composers=comp_list), :Year_median);

In [ ]:
plt_div2 =plot_composer_boxplot(pieces_df, :key_diversity, comp_list, comp_names;
    size=(1600, 800),
    title="",
    xtickfontsize=18,
    ytickfontsize=27,
    xlabelfontsize=25,
    xlabel="Key Diversity",
    ylabel="",
    color=cols[2]
    )

In [ ]:
savefig(plt_div2, "Key_Diversity_by_Composer.png")

# Key Diversity Time-Series

In [ ]:
df_kdiv = smooth_ts_feat(pieces_df, "key_diversity", y_ws = 30);

In [ ]:
plot_metric(df_kdiv, ylims=(0.5, 1.4), colors=cols[2:3], linewidth=4,

)
scatter!(compo_stats[!, :Year_median],
    compo_stats[!, :key_diversity_median],
    #yerror=[df_compval[ix_c,:Lowb_Div] df_compval[ix_c,:Topb_Div]], 
    ms=7, color=cols[2], alpha=0.8,
    label=""
)

x_ann = [1650, 1725, 1780, 1855, 1920]
y_ann = 1.29 .* ones(length(x_ann))
p_ann = ["I", "II", "III", "IV", "V"]
# Add labels with random offsets
labels = 1:nrow(compo_stats)  # or any array of numbers/letters
x_offset = 5 .* [1, 1, 1, -1, 1, -1, -1, 1, 1, 1, 1, 1, 1, 1] 
y_offset = 0.02 .* [1, 1, -1, 1, -1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

annotate!(compo_stats[!, :Year_median] .+ x_offset,
    compo_stats[!, :key_diversity_median] .+ y_offset,
    text.(string.(labels), 18, :black))

annotate!(x_ann, y_ann, text.(p_ann, 40, RGBA(0.5, 0.5, 0.5, 0.4)))

plot!(ylabel="Key Diversity", xlabel="Year",
    title="",
    legend=false,
    titlefontsize=18,
    xlabelfontsize=27,
    xtickfontsize=19,
    ylabelfontsize=27,
    ytickfontsize=19,
    left_margin=30Plots.mm
 ) # Add left margin)

In [ ]:
savefig("Key_Diversity_Timeseries.png")

# Key Uncertainty

In [ ]:
df_unc = smooth_ts_feat(pieces_df, "mean_unc", y_ws = 30);

In [ ]:
names(compo_stats)

In [ ]:
plot_metric(df_unc, ylims=(0.7, 1.4), colors=cols[1:2], linewidth=4)
scatter!(compo_stats[!, :Year_median],
    compo_stats[!, :mean_unc_median],
    #yerror=[df_compval[ix_c,:Lowb_Div] df_compval[ix_c,:Topb_Div]], 
    ms=7, color=cols[1], alpha=0.8,
    label=""
)

x_ann = [1650, 1725, 1780, 1855, 1920]
y_ann = 1.32 .* ones(length(x_ann))
p_ann = ["I", "II", "III", "IV", "V"]
# Add labels with offsets
labels = 1:nrow(compo_stats) 
# offsets are selected to minimize overlap
x_offset = 5 .* [1, 1, 1, -1, 1, -1, -1, 1, 1, 1, 1, 1, 1, 1]
y_offset = 0.02 .* [1, 1, -1, 1, -1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

annotate!(compo_stats[!, :Year_median] .+ x_offset,
    compo_stats[!, :mean_unc_median] .+ y_offset,
    text.(string.(labels), 18, :black))

annotate!(x_ann, y_ann, text.(p_ann, 40, RGBA(0.5, 0.5, 0.5, 0.4)))

plot!(ylabel="Key Uncertainty", xlabel="Year",
    title="",
    legend=false,
    titlefontsize=18,
    xlabelfontsize=27,
    xtickfontsize=19,
    ylabelfontsize=27,
    ytickfontsize=19,
    left_margin=20Plots.mm
) # Add left margin)

In [ ]:
savefig("Key_Uncertainty_Timeseries.png")

In [ ]:
plot_composer_boxplot(pieces_df, :mean_unc, comp_list, comp_names;
    size=(1600, 800),
    title="",
    xtickfontsize=15,
    ytickfontsize=27,
    xlabelfontsize=25,
    xlabel="Key Uncertainty",
    ylabel="",
    color=cols[1]
)

In [ ]:
savefig("Key_Uncertainty_by_Composer.png")

# Uncertainty vs Diversity

In [ ]:
function get_covmat(x, y)
    Σ = [cov(x, x) cov(x, y); cov(y, x) cov(y, y)]
    return Σ
end
mks = [:circle, :cross, :diamond, :hexagon, :rect, :pentagon, :star2, :utriangle];

In [ ]:
#kdf_pieces = CSV.read("KDF_Selection.csv", DataFrame);

In [ ]:
hh_d = @select(quads[1], :key_diversity)[!,1]
hh_u = @select(quads[1], :median_unc)[!,1]
hl_d = @select(quads[2], :key_diversity)[!,1]
hl_u = @select(quads[2], :median_unc)[!,1]
lh_d = @select(quads[3], :key_diversity)[!,1]
lh_u = @select(quads[3], :median_unc)[!,1]
ll_d = @select(quads[4], :key_diversity)[!,1]
ll_u = @select(quads[4], :median_unc)[!,1]

num_xt = 3
d_ext = vcat(hh_d[1:num_xt], hl_d[1:num_xt], lh_d[1:num_xt], ll_d[1:num_xt])
u_ext = vcat(hh_u[1:num_xt], hl_u[1:num_xt], lh_u[1:num_xt], ll_u[1:num_xt]);

In [ ]:
using Plots.PlotMeasures
# Add labels with offsets
label_numbers = 1:1:length(d_ext)
# offsets are selected to minimize overlap
x_offset = 0.035 .* [1, 1, 1, 1, 1, -1, 1, 1, 1, 1, 1, -1]
y_offset = 0.04 .* [1, 1, -1, 1, -1, -1, -1, 1, 1, -1, 1, -1]

pts = plot(xlabel="Diversity", ylabel="Uncertainty", size=(1200, 1000), background_color=RGBA(1, 1, 1, 0),
    legendfontsize=16,
    xtickfontsize=20,
    ytickfontsize=20,
    xlabelfontsize=25,
    ylabelfontsize=25,
    left_margin=20Plots.mm,
    foreground_color_legend=nothing
);

# plotting the x,y values for div,unc respectively.
for i in 1:length(m_per_names)-1
    df_p = @subset(pieces_df, :Period .== m_per_names[i])
    x_m = median(df_p[!, :key_diversity])
    y_m = median(df_p[!, :median_unc])
    cov_mat = get_covmat(df_p[!, :key_diversity], df_p[!, :median_unc])
    scatter!(pts, df_p[!, :key_diversity], df_p[!, :median_unc], m=mks[i], ms=5, color=cols[i], alpha=0.5, label=m_per_names[i])
    covellipse!([x_m, y_m], cov_mat, n_std=2, label="", linealpha=0.7, alpha=0, linecolor=cols[i], linewidth=3)
end

# plotting the circles surrounding the specific pieces near extreme values
for (x, y) in zip(d_ext,u_ext)
    θ = range(0, 2π, length=100)
    r = 0.02  # adjust this radius to match your data scale
    circle_x = x .+ r .* cos.(θ)
    circle_y = y .+ r .* sin.(θ)
    plot!(pts, circle_x, circle_y,
        seriestype=:path,
        linecolor=:black,
        linewidth=3,
        label="")
end

annotate!(d_ext .+ x_offset,
    u_ext .+ y_offset,
    text.(string.(label_numbers), 18, :black))

plot(pts)

In [ ]:
vcat([@select(quads[1],:Piece_ID,:key_diversity, :median_unc, :Composer, :Year, :NameFile)[1:num_xt,:], @select(quads[2],:Piece_ID,:key_diversity, :median_unc, :Composer, :Year, :NameFile)[1:num_xt,:], 
    @select(quads[3],:Piece_ID,:key_diversity, :median_unc, :Composer, :Year,:NameFile)[1:num_xt,:], 
    @select(quads[4],:Piece_ID,:key_diversity, :median_unc, :Composer, :Year, :NameFile)[1:num_xt,:]]...)

In [ ]:
@select(quads[3],:Composer, :NameFile, :Year, :key_diversity, :median_unc)

In [ ]:
savefig(pts, "Diversity_vs_Uncertainty.png")

In [ ]:
CSV.write("Pieces_data.csv", new_df)

## Plotting Novelties

In [ ]:
plot_composer_boxplot(pieces_df, :Novelty_numerals, comp_list, comp_names;
    size=(1600, 800),
    title="",
    xtickfontsize=15,
    ytickfontsize=27,
    xlims=(0.5,2.0),
    xlabelfontsize=25,
    xlabel="Novelty",
    ylabel="",
    color=cols[4]
)

In [ ]:
savefig("Novelty_by_Composer.png")

In [ ]:
compo_stats

In [ ]:
df_nov = smooth_ts_feat(pieces_df, "Novelty_original", y_ws = 30)
df_nov_num = smooth_ts_feat(pieces_df, "Novelty_numerals", y_ws = 30);

In [ ]:
plot_metric(df_nov,df2=df_nov_num, ylims=(0.35, 1.4), colors=[cols[5],cols[4]], linewidth=4, labels=["Original", "Transposed"], legendfontsize=16)
scatter!(compo_stats[!, :Year_median],
    compo_stats[!, :Novelty_numerals_median],
    #yerror=[df_compval[ix_c,:Lowb_Div] df_compval[ix_c,:Topb_Div]], 
    ms=7, color=cols[4], alpha=0.8,
    label=""
)

x_ann = [1650, 1725, 1780, 1855, 1920]
y_ann = 1.32 .* ones(length(x_ann))
p_ann = ["I", "II", "III", "IV", "V"]
# Add labels with offsets
labels = 1:nrow(compo_stats)
# offsets are selected to minimize overlap
x_offset = 5 .* [1, 1, 1, 1, -1, 1, -1, 1, 1, 1, 1, 1, 1, 1]
y_offset = 0.03 .* [1, 1, -1, 1, -1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

annotate!(compo_stats[!, :Year_median] .+ x_offset,
    compo_stats[!, :Novelty_numerals_median] .+ y_offset,
    text.(string.(labels), 18, :black))

annotate!(x_ann, y_ann, text.(p_ann, 40, RGBA(0.5, 0.5, 0.5, 0.4)))

plot!(ylabel="Novelty", xlabel="Year",
    title="",
    legend=(0.13, 0.8),
    #legend=false,
    titlefontsize=18,
    xlabelfontsize=27,
    xtickfontsize=19,
    ylabelfontsize=27,
    ytickfontsize=19,
    left_margin=15Plots.mm
    )

In [ ]:
savefig("Novelty_Timeseries.png")

In [ ]:
CSV.write("Pieces_Metrics.csv", pieces_df)

## Major and Minor Key Innovation

In [ ]:
minor_df = @subset(pieces_df, :Is_Minor .== 1)
major_df = @subset(pieces_df, :Is_Minor .== 0);

In [ ]:
df_novnum_minor = smooth_ts_feat(minor_df, "Novelty_numerals", y_ws = 30)
df_novor_minor = smooth_ts_feat(minor_df, "Novelty_original", y_ws = 30)
df_novnum_major = smooth_ts_feat(major_df, "Novelty_numerals", y_ws = 30)
df_novor_major = smooth_ts_feat(major_df, "Novelty_original", y_ws = 30);

In [ ]:
plot_metric(df_novor_major, df2=df_novor_minor, ylims=(0.35, 1.4), colors=[:red, cols[8]], linewidth=4, labels=["Major", "Minor"], legendfontsize=16)
plot!(df_nov[!, :Year], df_nov[!, :Med],
    linewidth=4,
    linestyle=:dash,
    fillalpha=0.2,
    color=cols[5],
    label="All"
)

x_ann = [1650, 1725, 1780, 1855, 1920]
y_ann = 1.32 .* ones(length(x_ann))
p_ann = ["I", "II", "III", "IV", "V"]
annotate!(x_ann, y_ann, text.(p_ann, 40, RGBA(0.5, 0.5, 0.5, 0.4)))
annotate!(1615, 1.35, text("a)", 45, :black))

plot!(ylabel="Novelty", xlabel="Year",
    title="",
    legend=(0.13, 0.8),
    #legend=false,
    titlefontsize=18,
    xlabelfontsize=27,
    xtickfontsize=19,
    ylabelfontsize=27,
    ytickfontsize=19,
    left_margin=15Plots.mm
)

In [ ]:
savefig("Novelty_MajMin_Original.png")

In [ ]:
plot_metric(df_novnum_major, df2=df_novnum_minor, ylims=(0.35, 1.4), colors=[:red, cols[8]], linewidth=4, labels=["Major", "Minor"], legendfontsize=16)
plot!(df_nov_num[!, :Year], df_nov_num[!, :Med],
    linewidth=4,
    linestyle=:dash,
    fillalpha=0.2,
    color=cols[5],
    label="All"
)

x_ann = [1650, 1725, 1780, 1855, 1920]
y_ann = 1.32 .* ones(length(x_ann))
p_ann = ["I", "II", "III", "IV", "V"]
annotate!(x_ann, y_ann, text.(p_ann, 40, RGBA(0.5, 0.5, 0.5, 0.4)))
annotate!(1615, 1.35, text("b)", 45, :black))

plot!(ylabel="Novelty", xlabel="Year",
    title="",
    #legend=(0.13, 0.8),
    legend=false,
    titlefontsize=18,
    xlabelfontsize=27,
    xtickfontsize=19,
    ylabelfontsize=27,
    ytickfontsize=19,
    left_margin=15Plots.mm
)

In [ ]:
savefig("Novelty_MajMin_Transposed.png")